In [1]:
from pathlib import Path

hr_dir = Path("/home/data/SyMTRS/hr")
night_dir = Path("/home/data/SyMTRS/night")

# Get just filenames (not full paths)
hr_files = {p.name for p in hr_dir.glob("*") if p.is_file()}
night_files = {p.name for p in night_dir.glob("*") if p.is_file()}

# Files in hr but missing in night
missing_in_night = sorted(hr_files - night_files)

print(f"Total HR images: {len(hr_files)}")
print(f"Total Night images: {len(night_files)}")
print(f"Missing in night: {len(missing_in_night)}\n")

for name in missing_in_night[:]:  # show first 20
    print(name)


Total HR images: 2070
Total Night images: 2070
Missing in night: 0



In [2]:
pip install -U pillow tqdm


  Using cached pillow-12.1.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
Using cached pillow-12.1.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.0 MB)
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pillow]2m1/2 [pillow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
visiofirm 1.1.1 requires Pillow==11.3.0, but you have pillow 12.1.0 which is incompatible.
visiofirm 1.1.1 requires tqdm==4.67.1, but you have tqdm 4.67.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
from PIL import Image, ImageFile
from tqdm import tqdm

# Helps PIL load truncated files (we still treat them as suspicious if verify/decode fails)
ImageFile.LOAD_TRUNCATED_IMAGES = True

ROOTS = [
    Path("/home/data/SyMTRS/hr"),
    Path("/home/data/SyMTRS/lr/x2"),
    Path("/home/data/SyMTRS/lr/x4"),
    Path("/home/data/SyMTRS/lr/x8"),
    Path("/home/data/SyMTRS/night"),
]

# Add/remove extensions as needed
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def is_image(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() in IMG_EXTS

def check_image(path: Path) -> tuple[bool, str]:
    """
    Returns (ok, reason). We do two checks:
    1) Image.verify() -> checks file integrity/headers
    2) full decode (load) -> catches some corruptions verify might miss
    """
    try:
        with Image.open(path) as im:
            im.verify()
        # Need to reopen after verify() (PIL requirement)
        with Image.open(path) as im:
            im.load()  # forces full decode
        return True, ""
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

def main():
    all_files = []
    for r in ROOTS:
        if not r.exists():
            print(f"[WARN] Missing folder: {r}")
            continue
        all_files.extend([p for p in r.rglob("*") if is_image(p)])

    print(f"Found {len(all_files)} image files to check.")

    bad = []
    for p in tqdm(all_files, desc="Checking", unit="img"):
        ok, reason = check_image(p)
        if not ok:
            bad.append((str(p), reason))

    print("\n=== RESULTS ===")
    print(f"Total checked: {len(all_files)}")
    print(f"Corrupted/unreadable: {len(bad)}")

    if bad:
        # Print a few examples
        print("\nFirst 20 bad files:")
        for path, reason in bad[:20]:
            print(f"- {path}\n  -> {reason}")

        # Save full report
        report_path = Path("corrupted_images_report.tsv")
        report_path.write_text(
            "path\treason\n" + "\n".join(f"{p}\t{r}" for p, r in bad),
            encoding="utf-8"
        )
        print(f"\nSaved full report to: {report_path.resolve()}")

if __name__ == "__main__":
    main()


Found 10350 image files to check.


Checking: 100%|██████████| 10350/10350 [07:21<00:00, 23.46img/s]


=== RESULTS ===
Total checked: 10350
Corrupted/unreadable: 0
